# Adaptive Blended Threshold — Pipeline v2 (WINDOW_SIZE=60, corrected k)

**Design correction versus v1**: `module3_pipeline/adaptive_threshold_blended.ipynb`
used a FIXED `K_ADAPTIVE=3.0` multiplier (`threshold = blended_mean + 3*blended_std`).
That value happened to work at window=30 because `mu_train + 3*sigma_train`
was coincidentally close to the true leak-free `val_p99` threshold there.
Testing the identical fixed-3.0 formula at window=60
(`experiments/window_size_60_full_pipeline.ipynb`) showed it does NOT
transfer — calibration got stuck at ~1.9-2.3% FPR against a 1% target, and
applying it made both `cc1_test` and `drift_cc2` WORSE than plain VAE-alone
scoring.

**The fix, built in from the start here**: derive `k` directly from the
model's own error distribution so the formula's global (no-local-history)
endpoint reproduces the empirical `val_p99` threshold exactly:

```
k = (val_p99 - mu_train) / sigma_train
```

Still fully leak-free — uses only `cc1_val`, same discipline as everywhere
else in this project. Verified in `experiments/window_size_60_threshold_fix.ipynb`
to restore correct calibration (FPR converges to ~0.99% instead of getting
stuck at ~1.9%) and to make the mechanism helpful again rather than harmful.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle, os
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from collections import deque

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
MODEL_DIR = os.path.join(BASE, 'models_v2')

WINDOW_SIZE = 60
BUFFER_SIZE = 500
K_ADAPTIVE_OLD_FIXED = 3.0   # kept only for reference/comparison prints below

SPLIT_FILES = {
    'cc1_val':   'cc1_val.csv',
    'cc1_test':  'cc1_test.csv',
    'drift_cc2': 'drift_complex_case2.csv',
}
DRIFT_SETS = ['drift_cc2']

print('Paths and constants configured.')

Paths and constants configured.


## Step 1 — Load model + prior results, derive the corrected `k`

In [2]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

meta       = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
prior_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
STATIC_VAL_P99 = prior_eval['thresholds']['val_p99']

GLOBAL_MEAN, GLOBAL_STD = meta['mu_train'], meta['sigma_train']
K_ADAPTIVE = (STATIC_VAL_P99 - GLOBAL_MEAN) / GLOBAL_STD

print(f'mu_train={GLOBAL_MEAN:.5f}  sigma_train={GLOBAL_STD:.5f}  val_p99={STATIC_VAL_P99:.5f}')
print(f'Corrected K_ADAPTIVE = {K_ADAPTIVE:.4f}  (v1 pipeline used a fixed 3.0 - see markdown above for why that breaks here)')

model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
model.eval()
CLIP = meta['clip']
print('Model loaded.')

mu_train=0.28407  sigma_train=0.24760  val_p99=1.44733
Corrected K_ADAPTIVE = 4.6982  (v1 pipeline used a fixed 3.0 - see markdown above for why that breaks here)
Model loaded.


## Step 2 — Load data + recover per-window `cmdb_id`

In [3]:
def window_meta(df, window_size, stride=1):
    cmdb_ids, end_ts, ys = [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ts     = g['timestamp'].values
        n = len(g)
        for i in range(0, n - window_size + 1, stride):
            if is_gap[i:i + window_size].any():
                continue
            cmdb_ids.append(cmdb_id)
            end_ts.append(ts[i + window_size - 1])
            ys.append(int(labels[i:i + window_size].any()))
    return np.array(cmdb_ids), np.array(end_ts), np.array(ys, dtype=np.int64)

cmdb_ids, end_ts, y, ft, mse = {}, {}, {}, {}, {}
for name, fname in SPLIT_FILES.items():
    df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
    df['is_gap'] = df['is_gap'].astype(bool)
    cmdb_ids[name], end_ts[name], y_replayed = window_meta(df, WINDOW_SIZE)

    y[name]  = np.load(os.path.join(WIN_DIR, f'y_{name}.npy'))
    ft[name] = np.load(os.path.join(WIN_DIR, f'ft_{name}.npy'), allow_pickle=True)
    Xr       = np.load(os.path.join(WIN_DIR, f'X_{name}.npy'))
    X        = np.clip(Xr, -CLIP, CLIP).astype(np.float32)
    mse[name] = model.anomaly_score(torch.from_numpy(X)).numpy()

    match = np.array_equal(y_replayed, y[name])
    print(f'  {name:10s}: {len(y[name]):>7,} windows  |  cmdb_id/order check: {"OK" if match else "MISMATCH!"}')
    assert match, f'{name}: ordering mismatch'

  cc1_val   :  20,763 windows  |  cmdb_id/order check: OK
  cc1_test  :  43,375 windows  |  cmdb_id/order check: OK
  drift_cc2 :  76,167 windows  |  cmdb_id/order check: OK


## Step 3 — Blended (shrinkage) adaptive-threshold algorithm, using the corrected `K_ADAPTIVE`

In [4]:
def run_blended(mse_arr, cmdb_arr, prior_strength, buffer_size=BUFFER_SIZE, k=K_ADAPTIVE,
                 global_mean=GLOBAL_MEAN, global_std=GLOBAL_STD):
    n = len(mse_arr)
    preds = np.zeros(n, dtype=np.int64)
    thresh_used = np.zeros(n, dtype=np.float64)
    weight_used = np.zeros(n, dtype=np.float64)
    buffers = {}

    for i in range(n):
        cid = cmdb_arr[i]
        buf = buffers.setdefault(cid, deque(maxlen=buffer_size))
        n_local = len(buf)
        w = n_local / (n_local + prior_strength)

        if n_local == 0:
            local_mean, local_std = global_mean, global_std
        else:
            arr = np.fromiter(buf, dtype=np.float64)
            local_mean, local_std = arr.mean(), arr.std()

        blended_mean = w * local_mean + (1 - w) * global_mean
        blended_std  = w * local_std  + (1 - w) * global_std
        t = blended_mean + k * blended_std

        thresh_used[i] = t
        weight_used[i] = w
        is_anom = mse_arr[i] > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse_arr[i])
    return preds, thresh_used, weight_used

print('Blended-threshold function defined (using corrected K_ADAPTIVE).')

Blended-threshold function defined (using corrected K_ADAPTIVE).


## Step 4 — Calibrate `PRIOR_STRENGTH` on `cc1_val`

In [5]:
PRIOR_CANDIDATES = [100, 500, 2000, 10000, 50000]
print(f'{"prior_strength":>14s} {"FPR on cc1_val":>16s}')
prior_fpr = {}
for ps in PRIOR_CANDIDATES:
    preds, _, _ = run_blended(mse['cc1_val'], cmdb_ids['cc1_val'], prior_strength=ps)
    fpr = preds.mean()
    prior_fpr[ps] = fpr
    print(f'{ps:14d} {fpr*100:15.2f}%')

BEST_PRIOR = min(PRIOR_CANDIDATES, key=lambda ps: abs(prior_fpr[ps] - 0.01))
print(f'\nChosen PRIOR_STRENGTH = {BEST_PRIOR}')

prior_strength   FPR on cc1_val
           100            1.32%
           500            1.10%
          2000            1.02%
         10000            1.00%
         50000            0.99%

Chosen PRIOR_STRENGTH = 10000


## Step 5 — Compare: static (`val_p99`) vs. blended adaptive

In [6]:
blended_results = {}
print(f'{"set":12s} {"method":10s} {"precision":>10s} {"recall":>8s} {"f1":>7s} {"TP":>5s} {"FP":>6s} {"FN":>5s} {"TN":>7s}')
for name in ['cc1_test'] + DRIFT_SETS:
    preds, thresh_used, weight_used = run_blended(mse[name], cmdb_ids[name], prior_strength=BEST_PRIOR)
    p = precision_score(y[name], preds, zero_division=0)
    r = recall_score(y[name], preds, zero_division=0)
    f1 = f1_score(y[name], preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y[name], preds).ravel()
    blended_results[name] = {'precision': p, 'recall': r, 'f1': f1, 'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
                              'thresh_used': thresh_used, 'weight_used': weight_used, 'preds': preds}
    print(f'{name:12s} {"blended":10s} {p:10.3f} {r:8.3f} {f1:7.3f} {tp:5d} {fp:6d} {fn:5d} {tn:7d}')

    static_pred = (mse[name] > STATIC_VAL_P99).astype(int)
    sp = precision_score(y[name], static_pred, zero_division=0)
    sr = recall_score(y[name], static_pred, zero_division=0)
    sf1 = f1_score(y[name], static_pred, zero_division=0)
    print(f'{name:12s} {"static":10s} {sp:10.3f} {sr:8.3f} {sf1:7.3f}')
    print(f'  -> blended vs static F1 change: {f1 - sf1:+.3f}\n')

set          method      precision   recall      f1    TP     FP    FN      TN
cc1_test     blended         0.986    0.848   0.912   217      3    39   43116
cc1_test     static          0.986    0.848   0.912
  -> blended vs static F1 change: +0.000

drift_cc2    blended         0.089    0.806   0.159   870   8960   210   66127
drift_cc2    static          0.086    0.800   0.155
  -> blended vs static F1 change: +0.005



## Step 6 — Per-fault-type recall, static vs. blended

In [7]:
per_fault_blended = {}
for name in ['cc1_test'] + DRIFT_SETS:
    per_fault_blended[name] = {}
    preds = blended_results[name]['preds']
    for ftype in sorted({v for v in ft[name] if isinstance(v, str)}):
        fmask = (ft[name] == ftype)
        n = int(fmask.sum())
        rec_blend = preds[fmask].mean() if n > 0 else float('nan')
        rec_static = ((mse[name] > STATIC_VAL_P99).astype(int))[fmask].mean() if n > 0 else float('nan')
        per_fault_blended[name][ftype] = {'n': n, 'recall': rec_blend}
        print(f'{name:12s} {ftype:14s} {n:5d} static={rec_static:.3f}  blended={rec_blend:.3f}')
    print()

cc1_test     cpu               88 static=0.761  blended=0.761
cc1_test     memory            76 static=0.868  blended=0.868
cc1_test     pod-failure       92 static=0.913  blended=0.913

drift_cc2    cpu              297 static=0.879  blended=0.882
drift_cc2    memory           594 static=0.852  blended=0.855
drift_cc2    pod-failure      189 static=0.513  blended=0.529



## Step 7 — Save results

In [8]:
save_results = {
    'buffer_size':      BUFFER_SIZE,
    'k_adaptive':        K_ADAPTIVE,
    'prior_strength':    BEST_PRIOR,
    'prior_fpr_search':  prior_fpr,
    'results':           {name: {k: v for k, v in r.items() if k not in ('thresh_used', 'weight_used', 'preds')} for name, r in blended_results.items()},
    'per_fault_recall':  per_fault_blended,
}
out_path = os.path.join(MODEL_DIR, 'vae_cc1_adaptive_blended_eval.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\models_v2\vae_cc1_adaptive_blended_eval.pkl


## Summary

The corrected `K_ADAPTIVE` (derived from `val_p99`, not assumed to be 3.0)
is what makes this mechanism work correctly at window=60 — see
`experiments/window_size_60_threshold_fix.ipynb` for the direct
before/after comparison proving this.